# 🚀 Super-Fast Multi-Agent Fusion Training (1,000 Samples)

This notebook performs end-to-end inference for all 4 agents on a small subset of 1,000 samples and trains the final Meta-Classifier. 

## Agents Included:
1.  **Spectral Agent** (XGBoost from Drive)
2.  **Prosodic Agent** (XGBoost from Drive)
3.  **Linguistic Agent** (Whisper-Tiny + Fine-tuned BERT from Drive)
4.  **SSL Agent** (JYP2024 WavLM-Base-Pruning from HuggingFace)

## Strategy:
*   **Speed over Quantity**: We process only 1,000 samples (500 Bonafide / 500 Spoof) to get the fusion model ready in minutes.
*   **Inference-Only**: We use the pre-trained/fine-tuned agents to get probability scores.
*   **Meta-Training**: We train a **Logistic Regression** meta-model on these scores.

In [ ]:
# Install dependencies for all agents
!pip install -q xgboost transformers datasets librosa soundfile imbalanced-learn joblib

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path
from google.colab import drive
import random

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Clone Repository
REPO_URL = "https://github.com/saltypal/Multi-Agent-Detection-of-AI-Generated-Speech"
BRANCH = "CoreDevelopment"

!rm -rf Multi-Agent-Detection-of-AI-Generated-Speech
!git clone -b {BRANCH} {REPO_URL}

# 3. Define Project Paths
PROJECT_ROOT = Path("/content/Multi-Agent-Detection-of-AI-Generated-Speech")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DRIVE_DATA = Path("/content/drive/MyDrive/40_PER_22_Data")

print(f"Project Root (Cloned): {PROJECT_ROOT}")
print(f"Data Directory (Drive): {DRIVE_DATA}")

In [ ]:
# ────────────────────────
# 4. Initialize All Agents
# ───────────📦 All agents loaded from Drive/HF
from spectral.spectral_model import SpectralAgent
from prosodic.prosodic_model import ProsodicAgent
from linguistic.linguistic_model import LinguisticAgent
from ssl.ssl_model import SSLAgent

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Initializing Agents on {device}...")

spec_agent = SpectralAgent(DRIVE_DATA / "spectral_results")
pros_agent = ProsodicAgent(DRIVE_DATA / "prosodic_results")
ling_agent = LinguisticAgent(DRIVE_DATA / "linguistic_bert_model", device=device)
ssl_agent  = SSLAgent("JYP2024/Wedefense_ASV2025_WavLM_Base_Pruning", device=device)

print("✅ All agents ready!")

In [ ]:
# ────────────────────
# 5. Sample 1,000 Audio Files
# ────────────────────
DATA_PATH = DRIVE_DATA / "train"

all_samples = []
for label, sub in [(0, "bonafide"), (1, "spoof")]:
    folder = DATA_PATH / sub
    if folder.exists():
        files = list(folder.glob("*.flac")) + list(folder.glob("*.wav"))
        sampled = random.sample(files, min(len(files), 500))
        for f in sampled:
            all_samples.append({'path': f, 'label': label})

random.shuffle(all_samples)
print(f"[*] Collected {len(all_samples)} samples.")

In [ ]:
# ────────────────────
# 6. Run Inference Pipeline
# ────────────────────
results = []

for item in tqdm(all_samples, desc="Agent Inference"):
    path = item['path']
    try:
        results.append({
            'P_spec': spec_agent.predict(path),
            'P_pros': pros_agent.predict(path),
            'P_ling': ling_agent.predict(path),
            'P_ssl':  ssl_agent.predict(path),
            'label': item['label']
        })
    except Exception as e:
        print(f"[!] Error on {path.name}: {e}")

fusion_df = pd.DataFrame(results)
fusion_df.to_csv(DRIVE_DATA / "fusion_1k_data.csv", index=False)
print(f"✅ Inference Done.")

In [ ]:
# ────────────────────
# 7. Train Meta-Classifier
# ────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import joblib

X = fusion_df[['P_spec', 'P_pros', 'P_ling', 'P_ssl']].values
y = fusion_df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

meta_model = LogisticRegression(class_weight='balanced')
meta_model.fit(X_train, y_train)

print("\n─── Meta-Agent Trust Weights ───")
for name, coef in zip(['Spectral', 'Prosodic', 'Linguistic', 'SSL'], meta_model.coef_[0]):
    print(f"{name:>10}: {coef:+.4f}")

y_prob = meta_model.predict_proba(X_test)[:, 1]
print(f"\nFinal Multi-Agent AUC: {roc_auc_score(y_test, y_prob):.4f}")

joblib.dump(meta_model, DRIVE_DATA / "fusion_meta_model.pkl")
print(f"✅ Saved Meta-Model to Drive.")